# Bert Pre-Training from scratch 

## Introduction

For the Project, because portuguese is grammarly complex, it's viable to consider a great custom grammatically correct tokenizer.

    - each word will be split into subwords (morfemas).

    - it will cover derivations, compositions, decompositions, suffix, prefix, male, female, plural, diminutive, aumentative, and much more.
    
    - the objective is to create one pre-trained comparable with BERTimbau and also Sabia3 (maritaca ai), but with a much less parameters (RoBERTinha). 

    - read DistilBERT!

    - therefore, it will be possible to host this LLM locally in low specification computer or maybe even mobile devices!

    - useful for some scenarios without internet and such. 

    - good portability

    - open source project, no cost !

    - custom tokenizator fit for portuguese

    - use accents (ç ã á ...)

    - keep all lower case (normalize noise) ? or make it case sensitive for more control ?

    - test this model in several generalistic tasks (GLUE), and also specific tasks such as leNER-br

Plano Brasileiro de IA (PBIA) 

    2024 - 2028

    Com um investimento previsto de R$ 23 bilhões em quatro anos, o plano ambicioso visa transformar o país em referência mundial em inovação e eficiência no uso da inteligência artificial, especialmente no setor público.

- Separar o dataset em train/test[30%], depois avaliar o modelo no test -> done!

- buscar mais modelos mais atualizados, [2020 - 2024]. -> ModernBERT

avaliar o modelo depois de treinado, avaliando no MLM, e também nas tarefas que eu desejo, (NER, Classificacao), e talvez GLUE em portugues

 - ver a literatura para saber como avaliar as tarefas desejadas. 

 - eventualmente comparar os resultados gerados do modelo com demais modelos da literatura

 - se valer a pena, trazer então as tecnicas desses modelos e incorporar no nosso

Treinar um modelo bem grande com muitos parametros depois de definir bem os demais detalhes:

- 1. dados a ser utilizados
- 2. tokenizer e context size
- 3. modelo (e avaliacao)
- 4. tarefas de fine-tuning (e avaliacao) 
- 5. comparar com outros modelos da literatura

Escrever um artigo cientifico

Squad e GLUE em portugues!

## Check for GPU and libraries:

In [1]:
# import some libraries
from tqdm import tqdm
from multiprocessing import cpu_count
from datasets import load_dataset, Dataset
import os, torch

# check enabled GPU
print(torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
   print(torch.cuda.get_device_properties(i))

True
_CudaDeviceProperties(name='Tesla V100S-PCIE-32GB', major=7, minor=0, total_memory=32501MB, multi_processor_count=80, uuid=ab6a4880-78de-c23c-7abc-e0f4ee2f44e3, L2_cache_size=6MB)
_CudaDeviceProperties(name='Tesla V100S-PCIE-32GB', major=7, minor=0, total_memory=32501MB, multi_processor_count=80, uuid=ab69db1d-56c6-9867-01a1-64e74ef3c648, L2_cache_size=6MB)


In [2]:
import transformers 

print(transformers.__version__)

4.48.1


## Dataset 

### Wikipedia pages (2023) (Portuguese)

- each document (page in wikipedia) is saved in a row 

- the wikipedia dataset has several columns, including title and etc. I only used the main text of the page

- downloading dataset from hugging face only needs to be done once, and it will be saved in cache for future loads!

https://huggingface.co/datasets/wikimedia/wikipedia

### C4 (2020) (Portuguese)

https://huggingface.co/datasets/allenai/c4

https://paperswithcode.com/dataset/c4

### Oscar

- noisy, avoid for now


### MC4 (legal)

- multilingual variant of the C4 dataset, composed of natural text drawn from the public Common Crawl Web Scrape.

- consider cleaning more this dataset, it is noisy

https://huggingface.co/datasets/joelniklaus/legal-mc4

### Multi_Legal_Pile (pt)

- is a large-scale multilingual legal dataset suited for pretraining language models. It spans over 24 languages and five legal text types.

- case_law dataset (Jurisprudência)

- very huge dataset, used only 5 %

https://huggingface.co/datasets/joelniklaus/Multi_Legal_Pile

### Lener-br dataset

- LeNER-Br is a dataset for named entity recognition (NER) in Brazilian Legal Text.

- Composed of 70 legal documents, manually curated and tagged

https://github.com/peluz/lener-br/


### Concatenate Datasets 

- skip mc4 for now (noisy)

- lener is very small compared with the others

- complete dataset is around 50% wikipedia, 50% legal docs

- 1.97 M documents

### Data preprocessing

Split Dataset documents by their paragraphs, to generate more rows

Each page will be split by their paragraphs, each one becoming a new document

Save it!

### Load it!

Separate dataset into train and test

*The splits are shuffled by default, but you can set shuffle=False to prevent shuffling.*

## Tokenizer 

In [3]:
vocabulary_size = 32_768
context_size = 512
language = "pt"

In [4]:
tokenizer_name = f"tokenizers/custom/{language}/{vocabulary_size:_}"

### Tokenizer Training

Train a tokenizer, from scratch! 

- it is important to train a tokenizer, because it is the responsible for representing the input data so the model is able to interact and interpret the vocabulary provided!

- each word used in the input, should have a token (or a sequence of tokens) that will be used to represent it, and become embeddings when interacting with the model

- the pre-loaded tokenizer is uncased, meaning that all uppercase letters will be converted to lowercase to reduce complexity and vocabulary size

It will contain the following special tokens:

    0. [PAD]: Padding (to fill empty spots)
    1. [UNK]: Unknown
    2. [CLS]: Classification (initial token used as classifier)
    3. [SEP]: Separator (for sequences) (and also it is the ending token)
    4. [MASK]: Masking (token that represents a masked token)

- It's important for a tokenizer to have a **good compression rate**, meaning that the sequence of tokens generated will be as small as possible. As we know that is computationally expensive to increase the context size (sequence token limit)

- Vocabulary size will help with the compression rate (low fertility), the higher the vocab size, the lower the need for splitting words into subtokens. 

#### Specials Tokens

#### Normalizer

#### Pre-Tokenizer

#### Post-Tokenizer

#### Decoder

#### Train tokenizer with unigram model

Deactivate tokenizer paralelism to avoid error when using built-in paralelism from hugging face: map batch processing and accelerator

#### Convert to a fast instance of tokenizer

Save the tokenizer: 

### Load Tokenizer

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, local_files_only=True)

tokenizer

PreTrainedTokenizerFast(name_or_path='tokenizers/custom/pt/32_768', vocab_size=32768, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

### Test tokenizer

#### Objetively Evaluate Tokenizer on Compression Rate (1/Fertility)

## Tokenize the dataset

The bert-base-uncased tokenizer was configured to use a {model_max_length} of 512, meaning that the sequence input given to the bert model can have up to 512 tokens (context size).

Nevertheless, for better performance (speed), I will hard limit if to 128:

### Examples

### Process the Dataset:

This next step is to manipulate the dataset for a suitable format for the model:

- That is, tokenize the dataset, and truncate the documents with more than {model_max_length} tokens. 

- This is done in the CPU in batches and parellized across the cores.

Save the dataset, now, already tokenized, locally. 

### Load Tokenized Datasets

## Filtering

In [6]:
filtered_datasets_name = f"dataset/filtered/custom/{language}/vocab_size:{vocabulary_size:_}/context_size:{context_size}"

### Filter Dataset

To train the bert model faster, we will optimize the documents used for training, that is, we will reduce the dataset to retain only good documents. 

**The criteria used**  *the documents will be filtered and only be valid when*:

    - it has more than 32 tokens.

    - it was not truncated

### Save

### Load

In [7]:
from datasets import load_from_disk

filtered_datasets = load_from_disk(filtered_datasets_name)

filtered_datasets

Loading dataset from disk:   0%|          | 0/82 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 11341505
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 1259916
    })
})

Make training dataset smaller for easier experimentation

In [8]:
training_dataset = filtered_datasets["train"] # .select([i for i in range( int(0.15 * len(filtered_datasets["train"])) )])

training_dataset

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 11341505
})

In [9]:
evaluation_dataset = filtered_datasets["test"].select([i for i in range( int(0.0008 * len(filtered_datasets["test"])) )])

evaluation_dataset

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 1007
})

convert to pytorch format and remove token type id, which is ignored by modernbert

In [10]:
training_dataset.set_format(type="pt", columns=['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'])

evaluation_dataset.set_format(type="pt", columns=['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'])

training_dataset, evaluation_dataset

(Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
     num_rows: 11341505
 }),
 Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
     num_rows: 1007
 }))

## Model Training

import a pre-configured bert and choose a name for the model

In [11]:
model_name = f"Modern/{2.2}"

In [12]:
from transformers import ModernBertConfig

config = ModernBertConfig.from_pretrained("answerdotai/ModernBERT-base", reference_compile=False)

config

ModernBertConfig {
  "_name_or_path": "ModernBERT-base",
  "architectures": [
    "ModernBertForMaskedLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 50281,
  "classifier_activation": "gelu",
  "classifier_bias": false,
  "classifier_dropout": 0.0,
  "classifier_pooling": "mean",
  "cls_token_id": 50281,
  "decoder_bias": true,
  "deterministic_flash_attn": false,
  "embedding_dropout": 0.0,
  "eos_token_id": 50282,
  "global_attn_every_n_layers": 3,
  "global_rope_theta": 160000.0,
  "gradient_checkpointing": false,
  "hidden_activation": "gelu",
  "hidden_size": 768,
  "initializer_cutoff_factor": 2.0,
  "initializer_range": 0.02,
  "intermediate_size": 1152,
  "layer_norm_eps": 1e-05,
  "local_attention": 128,
  "local_rope_theta": 10000.0,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "mlp_dropout": 0.0,
  "model_type": "modernbert",
  "norm_bias": false,
  "norm_eps": 1e-05,
  "num_attention_heads": 12,
  "num_hidden_layers": 22,
  "pa

In [13]:
# diminish the specs

# {bert-small} with max_position_embedding = 128

config.vocab_size = vocabulary_size
config.num_hidden_layers = 16
config.num_attention_heads = 8
config.intermediate_size = 1024
config.hidden_size = 512
config.max_position_embeddings = (context_size*3)//2
config.pad_token_id = 0
config.bos_token_id = 2
config.cls_token_id = 2
config.eos_token_id = 3
config.sep_token_id = 3

config

ModernBertConfig {
  "_name_or_path": "ModernBERT-base",
  "architectures": [
    "ModernBertForMaskedLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 2,
  "classifier_activation": "gelu",
  "classifier_bias": false,
  "classifier_dropout": 0.0,
  "classifier_pooling": "mean",
  "cls_token_id": 2,
  "decoder_bias": true,
  "deterministic_flash_attn": false,
  "embedding_dropout": 0.0,
  "eos_token_id": 3,
  "global_attn_every_n_layers": 3,
  "global_rope_theta": 160000.0,
  "gradient_checkpointing": false,
  "hidden_activation": "gelu",
  "hidden_size": 512,
  "initializer_cutoff_factor": 2.0,
  "initializer_range": 0.02,
  "intermediate_size": 1024,
  "layer_norm_eps": 1e-05,
  "local_attention": 128,
  "local_rope_theta": 10000.0,
  "max_position_embeddings": 768,
  "mlp_bias": false,
  "mlp_dropout": 0.0,
  "model_type": "modernbert",
  "norm_bias": false,
  "norm_eps": 1e-05,
  "num_attention_heads": 8,
  "num_hidden_layers": 16,
  "pad_token_id": 0

Train for MLM task

In [14]:
from transformers import ModernBertForMaskedLM

model = ModernBertForMaskedLM(config=config)

print("parameters: ", model.num_parameters())

model

parameters:  59032576


ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(32768, 512, padding_idx=0)
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=512, out_features=1536, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=512, out_features=512, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=512, out_features=2048, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1024, out_features=512, bias=False)
        )
      )
      (1-15): 15

to mask the input randomly, we will use a data collator, which will randomly tokenize 30% of the input as [MASK] 

it is a dynamical masking (such as the one used in roBERTa), meaning that each time it a document is given as the input in the model, as random porcentile will the masked. 

in static masking, all the dataset is prior masked, meaning that the masked tokens is predefinied and if you run multiple epochs, it won't make a difference. 

In [15]:
from transformers import DataCollatorForLanguageModeling

# mask 30% of the tokens
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm = True,
    mlm_probability=0.3
)

As for the training arguments, 

- we will train the model passing through the dataset twice (epoch == 2)

- the batch size (number of inputs trained per iteration) is 128

- RoBERTa proved that bigger batches will output better results, that why we also use gradient accumulation steps 

- only after 4 {gradient_accumulation_steps}, the model will backpropagate and update parameters.

- each 1_000 iterations (steps) the model parameters will be locally saved

- the number of saves or checkpoints saved simultaneusly is 3

- it will used mixed precision of fp16, meaning that sometimes (when the precision won't affect the accuracy),
it will not use the float of 32 bits and instead use float of 16 bits, speeding up the training process

In [16]:
import evaluate

classification_metrics = evaluate.combine(["accuracy"])

def compute_metrics_modern(eval_pred):

    labels = eval_pred.label_ids
    preds = eval_pred.predictions.argmax(-1)

    return classification_metrics.compute(predictions=preds.flatten(), references=labels.flatten())

In [17]:
torch.cuda.empty_cache()

## Testing the Model

Load the self trained model and create a pipeline that will:

- tokenize the input
 
- push to the model

- output the top 5 token that the model predict to fill each masked token

In [18]:
from transformers import pipeline, ModernBertForMaskedLM

model = ModernBertForMaskedLM.from_pretrained(f"trained/{model_name}")

In [19]:
from transformers import Trainer, TrainingArguments
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics_eval(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    indices = [[i for i, x in enumerate(labels[row]) if x != -100] for row in range(len(labels))]

    labels = [labels[row][indices[row]] for row in range(len(labels))]
    labels = [item for sublist in labels for item in sublist]

    predictions = [predictions[row][indices[row]] for row in range(len(predictions))]
    predictions = [item for sublist in predictions for item in sublist]

    results = metric.compute(predictions=predictions, references=labels)
    results["eval_accuracy"] = results["accuracy"]
    print(results)
    results.pop("accuracy")

    return results

evaluation_args = TrainingArguments(
    output_dir=f'evaluating/{model_name}',
    num_train_epochs=1,                     # number of training epochs
    eval_strategy = "steps",                 # do NOT evaluate during training

    gradient_accumulation_steps = 1,
    eval_accumulation_steps=1,
    per_device_train_batch_size=32,          # batch size for training
    per_device_eval_batch_size=32,           # batch size for evaluation

    # logging_dir=f'training-logs/{model_name}',                # directory for storing logs
    
    logging_first_step=True, # output the initial loss
    logging_strategy="steps",
    logging_steps=100,

    save_strategy="steps",
    save_steps=100,                      # Save checkpoints every 100 steps
    save_total_limit=5,                  # Limit the total number of saved checkpoints

    fp16=True,                            # Enable mixed precision for faster training
)

evaluator = Trainer(
    model=model,   
    args = evaluation_args,
    eval_dataset=evaluation_dataset,         # Evaluation dataset
    compute_metrics=compute_metrics_eval,    # Function to compute metrics during evaluation
    data_collator=data_collator,
)


# Evaluate the model
eval_results = evaluator.evaluate()

# Print the results
print(eval_results)

/home/wallacelw/miniconda3/envs/modernBERT/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'accuracy': 0.6574511819116136, 'eval_accuracy': 0.6574511819116136}
{'eval_accuracy': 0.6574511819116136, 'eval_loss': 1.817528486251831, 'eval_runtime': 316.8563, 'eval_samples_per_second': 3.178, 'eval_steps_per_second': 0.05}


In [20]:
fill_mask = pipeline(
    "fill-mask",
    model=model,
    tokenizer=tokenizer
)

test1 = "Eu não entendi [MASK], como proceder."

out = tokenizer(test1)

out

Device set to use cuda:0


{'input_ids': [2, 5, 12, 18, 53, 20849, 22, 4, 5, 7, 39, 10019, 29, 5, 11, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [21]:
tokenizer.convert_ids_to_tokens(out['input_ids'])

['[CLS]',
 '▁',
 'e',
 'u',
 '▁não',
 '▁entend',
 'i',
 '[MASK]',
 '▁',
 ',',
 '▁como',
 '▁procede',
 'r',
 '▁',
 '.',
 '[SEP]']

In [22]:
fill_mask(test1)

[{'score': 0.19091445207595825,
  'token': 288,
  'token_str': 'isso',
  'sequence': 'eu não entendi isso , como proceder .'},
 {'score': 0.11372603476047516,
  'token': 10,
  'token_str': 'o',
  'sequence': 'eu não entendio , como proceder .'},
 {'score': 0.11161330342292786,
  'token': 1608,
  'token_str': 'nada',
  'sequence': 'eu não entendi nada , como proceder .'},
 {'score': 0.10827707499265671,
  'token': 8,
  'token_str': 'a',
  'sequence': 'eu não entendia , como proceder .'},
 {'score': 0.05282294750213623,
  'token': 21,
  'token_str': 'que',
  'sequence': 'eu não entendi que , como proceder .'}]

In [23]:
fill_mask("Atiraram o pau no [MASK]")

[{'score': 0.15090312063694,
  'token': 7228,
  'token_str': 'peito',
  'sequence': 'atiraram o pau no peito'},
 {'score': 0.031925950199365616,
  'token': 163,
  'token_str': 'final',
  'sequence': 'atiraram o pau no final'},
 {'score': 0.03070329874753952,
  'token': 13483,
  'token_str': 'ombro',
  'sequence': 'atiraram o pau no ombro'},
 {'score': 0.027167806401848793,
  'token': 10947,
  'token_str': 'ringue',
  'sequence': 'atiraram o pau no ringue'},
 {'score': 0.01995064690709114,
  'token': 2921,
  'token_str': 'gelo',
  'sequence': 'atiraram o pau no gelo'}]

In [24]:
fill_mask("Essa frase tem [MASK] tokens, portanto ele vai gerar dois [MASK]")

[[{'score': 0.38997483253479004,
   'token': 129,
   'token_str': 'dois',
   'sequence': '[CLS] essa frase tem dois tokens , portanto ele vai gerar dois[MASK][SEP]'},
  {'score': 0.0697893276810646,
   'token': 171,
   'token_str': 'três',
   'sequence': '[CLS] essa frase tem três tokens , portanto ele vai gerar dois[MASK][SEP]'},
  {'score': 0.040525197982788086,
   'token': 300,
   'token_str': 'quatro',
   'sequence': '[CLS] essa frase tem quatro tokens , portanto ele vai gerar dois[MASK][SEP]'},
  {'score': 0.02289964258670807,
   'token': 191,
   'token_str': 'duas',
   'sequence': '[CLS] essa frase tem duas tokens , portanto ele vai gerar dois[MASK][SEP]'},
  {'score': 0.01989103853702545,
   'token': 173,
   'token_str': '10',
   'sequence': '[CLS] essa frase tem 10 tokens , portanto ele vai gerar dois[MASK][SEP]'}],
 [{'score': 0.25679802894592285,
   'token': 544,
   'token_str': 'pontos',
   'sequence': '[CLS] essa frase tem[MASK] tokens , portanto ele vai gerar dois pontos[S

## NER task

### Fine-tuning for for NER task

Download the curated dataset for portuguese NER task in the legal scope.

It already has the train, validation and test splits

In [25]:
from datasets import load_dataset

lenerNER = load_dataset("peluz/lener_br", 
                        trust_remote_code=True, 
                        num_proc=cpu_count())

lenerNER

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1390
    })
})

In [26]:
print( lenerNER["train"][0]["id"] ) # id of the document [0, 70)
    
print( lenerNER["train"][0]["tokens"] ) # words

print( lenerNER["train"][0]["ner_tags"] ) # tags for the words

0
['EMENTA', ':', 'APELAÇÃO', 'CÍVEL', '-', 'AÇÃO', 'DE', 'INDENIZAÇÃO', 'POR', 'DANOS', 'MORAIS', '-', 'PRELIMINAR', '-', 'ARGUIDA', 'PELO', 'MINISTÉRIO', 'PÚBLICO', 'EM', 'GRAU', 'RECURSAL', '-', 'NULIDADE', '-', 'AUSÊNCIA', 'DE', 'INTERVENÇÃO', 'DO', 'PARQUET', 'NA', 'INSTÂNCIA', 'A', 'QUO', '-', 'PRESENÇA', 'DE', 'INCAPAZ', '-', 'PREJUÍZO', 'EXISTENTE', '-', 'PRELIMINAR', 'ACOLHIDA', '-', 'NULIDADE', 'RECONHECIDA', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Each ner_tag id corresponds to one of the following labels:

**BIO notation:**

B - means that is the beginning of a entity

I - means that it inside the entity (after a B tag)

O - means that this token a non-entity tokens

In [27]:
label_list = lenerNER["train"].features[f"ner_tags"].feature.names

label_list

['O',
 'B-ORGANIZACAO',
 'I-ORGANIZACAO',
 'B-PESSOA',
 'I-PESSOA',
 'B-TEMPO',
 'I-TEMPO',
 'B-LOCAL',
 'I-LOCAL',
 'B-LEGISLACAO',
 'I-LEGISLACAO',
 'B-JURISPRUDENCIA',
 'I-JURISPRUDENCIA']

Each word can be split into several tokens by our pre trained tokenizer, let's see an example:

word "PARQUET" was split into 2 tokens

In [28]:
exampleNER = lenerNER["train"][0]

example_tokenized_input = tokenizer(exampleNER["tokens"], is_split_into_words = True)

example_tokens = tokenizer.convert_ids_to_tokens(example_tokenized_input["input_ids"])

print(exampleNER["tokens"])
print(example_tokenized_input["input_ids"])
print(example_tokens)
print(exampleNER["ner_tags"])

['EMENTA', ':', 'APELAÇÃO', 'CÍVEL', '-', 'AÇÃO', 'DE', 'INDENIZAÇÃO', 'POR', 'DANOS', 'MORAIS', '-', 'PRELIMINAR', '-', 'ARGUIDA', 'PELO', 'MINISTÉRIO', 'PÚBLICO', 'EM', 'GRAU', 'RECURSAL', '-', 'NULIDADE', '-', 'AUSÊNCIA', 'DE', 'INTERVENÇÃO', 'DO', 'PARQUET', 'NA', 'INSTÂNCIA', 'A', 'QUO', '-', 'PRESENÇA', 'DE', 'INCAPAZ', '-', 'PREJUÍZO', 'EXISTENTE', '-', 'PRELIMINAR', 'ACOLHIDA', '-', 'NULIDADE', 'RECONHECIDA', '.']
[2, 14, 5095, 5, 38, 4690, 409, 5, 42, 1979, 5, 17, 5, 409, 9, 14582, 43, 36, 3758, 6, 6467, 5, 17, 10086, 5, 17, 6517, 18, 783, 66, 5, 1180, 402, 14, 2056, 7420, 29, 6, 8, 41, 5, 17, 5, 37, 18, 41, 177, 5, 17, 5, 8, 18, 4629, 9, 5, 4472, 15, 840, 40, 31, 5, 22, 13990, 5, 8, 5705, 10, 5, 17, 5, 1046, 9, 6949, 83, 5, 17, 612, 3403, 10, 5152, 5, 17, 10086, 5, 8, 19357, 5, 17, 5, 37, 18, 41, 177, 3907, 5, 11, 3]
['[CLS]', '▁em', 'enta', '▁', ':', '▁apel', 'ação', '▁', 'c', 'ível', '▁', '-', '▁', 'ação', '▁de', '▁indeniza', 'ção', '▁por', '▁dano', 's', '▁morais', '▁', '-'

**word_id:** the index of the word given prior to the tokenization!

check the example with word index 28

In [29]:
# if a word is split into 2 tokens, both will share the same word_id

example_word_ids = example_tokenized_input.word_ids()

for i, j in zip(example_word_ids, example_tokens):
    print(i, j)

None [CLS]
0 ▁em
0 enta
1 ▁
1 :
2 ▁apel
2 ação
3 ▁
3 c
3 ível
4 ▁
4 -
5 ▁
5 ação
6 ▁de
7 ▁indeniza
7 ção
8 ▁por
9 ▁dano
9 s
10 ▁morais
11 ▁
11 -
12 ▁preliminar
13 ▁
13 -
14 ▁arg
14 u
14 ida
15 ▁pelo
16 ▁
16 ministério
17 ▁público
18 ▁em
19 ▁grau
20 ▁recu
20 r
20 s
20 a
20 l
21 ▁
21 -
22 ▁
22 n
22 u
22 l
22 idade
23 ▁
23 -
24 ▁
24 a
24 u
24 sência
25 ▁de
26 ▁
26 intervenção
27 ▁do
28 ▁parque
28 t
29 ▁na
30 ▁
30 i
30 nstância
31 ▁
31 a
32 ▁qu
32 o
33 ▁
33 -
34 ▁
34 presença
35 ▁de
36 ▁incapa
36 z
37 ▁
37 -
38 ▁pre
38 juíz
38 o
39 ▁existente
40 ▁
40 -
41 ▁preliminar
42 ▁
42 a
42 colhida
43 ▁
43 -
44 ▁
44 n
44 u
44 l
44 idade
45 ▁reconhecida
46 ▁
46 .
None [SEP]


Tokenize the input and preprocess. 

We need also to align, because some words were split into several tokens

label of [CLS], [SEP] and for tokens that are suffixes (##) have label == -100 (meaning that it will be ignored for the ner tag prediction)

In [30]:
def tokenize_and_align_labels(examples):
    
    tokenized_inputs = tokenizer(examples["tokens"], 
                                 max_length=context_size,
                                 truncation=True, 
                                 is_split_into_words=True)

    labels = []

    for i, label in enumerate(examples[f"ner_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None: # [CLS], [SEP]
                label_ids.append(-100)

            elif word_idx != previous_word_idx: # new word
                label_ids.append(label[word_idx])

            else: # suffix of a previous token
                label_ids.append(-100)

            previous_word_idx = word_idx
            
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

tokenized_lenerNER = lenerNER.map(tokenize_and_align_labels,
                                  batched=True,
                                  num_proc=cpu_count(),
                                  )

Map (num_proc=96):   0%|          | 0/7828 [00:00<?, ? examples/s]

Map (num_proc=96):   0%|          | 0/1177 [00:00<?, ? examples/s]

Map (num_proc=96):   0%|          | 0/1390 [00:00<?, ? examples/s]

In [31]:
tokenized_lenerNER

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7828
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1177
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1390
    })
})

In [32]:
from transformers import DataCollatorForTokenClassification

ner_collator = DataCollatorForTokenClassification(tokenizer=tokenizer,
                                                  padding=True,
                                                  )

load a evaluation metric that will be used to compute the total evaluation for a sequence (seq eval)

In [33]:
import evaluate

seqeval = evaluate.load("seqeval")

define the method to compute metrics (precision, recall, f1, accuracy)

In [34]:
import numpy as np

def compute_metrics_ner(p):

    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
}

Create dictionaries to convert id to label and vice-versa

In [35]:
id2label = {}
label2id = {}

for i, lab in enumerate(label_list):

    id2label[i] = lab
    label2id[lab] = i

print(id2label)
print(label2id)

{0: 'O', 1: 'B-ORGANIZACAO', 2: 'I-ORGANIZACAO', 3: 'B-PESSOA', 4: 'I-PESSOA', 5: 'B-TEMPO', 6: 'I-TEMPO', 7: 'B-LOCAL', 8: 'I-LOCAL', 9: 'B-LEGISLACAO', 10: 'I-LEGISLACAO', 11: 'B-JURISPRUDENCIA', 12: 'I-JURISPRUDENCIA'}
{'O': 0, 'B-ORGANIZACAO': 1, 'I-ORGANIZACAO': 2, 'B-PESSOA': 3, 'I-PESSOA': 4, 'B-TEMPO': 5, 'I-TEMPO': 6, 'B-LOCAL': 7, 'I-LOCAL': 8, 'B-LEGISLACAO': 9, 'I-LEGISLACAO': 10, 'B-JURISPRUDENCIA': 11, 'I-JURISPRUDENCIA': 12}


load the pre trained model of bert to fine tune it for NER task, attaching a Token Classification HEAD

In [36]:
from transformers import ModernBertForTokenClassification

ner_model = ModernBertForTokenClassification.from_pretrained(
    pretrained_model_name_or_path=f"trained/{model_name}",
    num_labels = len(label_list),
    id2label = id2label,
    label2id= label2id
)

ner_model

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at trained/Modern/2.2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ModernBertForTokenClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(32768, 512, padding_idx=0)
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=512, out_features=1536, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=512, out_features=512, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=512, out_features=2048, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1024, out_features=512, bias=False)
        )
      )
     

Train it in batches of size 16, while also computing the evaluation for the validation set

In [37]:
from transformers import TrainingArguments

ner_training_args = TrainingArguments(
    output_dir = f"trained/NER/{model_name}",
    overwrite_output_dir=True,
    
    num_train_epochs = 2,

    eval_strategy = "epoch",

    gradient_accumulation_steps = 1,
    eval_accumulation_steps = 1,

    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    
    learning_rate = 2e-5,
    weight_decay = 0.01,
    
    logging_first_step=True,
    save_strategy = "steps",
    save_steps=100,                      # Save checkpoints every 100 steps
    save_total_limit=5,

    fp16=True,
)

In [38]:
from transformers import Trainer

ner_trainer = Trainer(
    model = ner_model,
    train_dataset = tokenized_lenerNER["train"],
    eval_dataset = tokenized_lenerNER["validation"],
    data_collator = ner_collator,
    compute_metrics = compute_metrics_ner,
    args = ner_training_args
)

In [39]:
ner_trainer.train()

/home/wallacelw/miniconda3/envs/modernBERT/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,2.083900,0.099537,0.688392,0.795930,0.738265,0.971457
2,2.083900,0.091122,0.715976,0.798680,0.755070,0.974469


/home/wallacelw/miniconda3/envs/modernBERT/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/wallacelw/miniconda3/envs/modernBERT/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/wallacelw/miniconda3/envs/modernBERT/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/wallacelw/miniconda3/envs/modernBERT/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return

TrainOutput(global_step=490, training_loss=0.08605617065818942, metrics={'train_runtime': 122.7412, 'train_samples_per_second': 127.553, 'train_steps_per_second': 3.992, 'total_flos': 1013462737411704.0, 'train_loss': 0.08605617065818942, 'epoch': 2.0})

### Test NER Task

In [40]:
modern_ner_name = f"trained/NER/{model_name}"

In [41]:
import pandas as pd

def tag_sentence(text:str):

    # convert our text to a  tokenized sequence
    inputs = tokenizer(text, truncation=True, return_tensors="pt", return_token_type_ids=False).to("cuda:0")

    # get outputs
    outputs = ner_model(**inputs)

    # convert to probabilities with softmax
    probs = outputs[0][0].softmax(1)

    # get the tags with the highest probability
    word_tags = [(tokenizer.decode(inputs['input_ids'][0][i].item()), id2label[tagid.item()]) 
                  for i, tagid in enumerate (probs.argmax(axis=1))]

    return pd.DataFrame(word_tags, columns=['word', 'tag'])

In [42]:
sample = "O senador de São Paulo decidiu faltar a apelação, movida pelo Supremo Tribunal Federal."

tag_sentence(sample)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


,word,tag
0,[CLS],O
1,,O
2,o,O
3,senador,O
4,de,O
5,são,I-ORGANIZACAO
6,paulo,I-ORGANIZACAO
7,de,O
8,cidiu,O
9,falta,O


Finally, evaluate the fine tuned model in the test dataset

In [43]:
ner_trainer.evaluate(eval_dataset=tokenized_lenerNER["test"])

/home/wallacelw/miniconda3/envs/modernBERT/lib/python3.12/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.0596013143658638,
 'eval_precision': 0.7483282674772036,
 'eval_recall': 0.8163129973474801,
 'eval_f1': 0.7808436409768474,
 'eval_accuracy': 0.9817849476078131,
 'eval_runtime': 5.1186,
 'eval_samples_per_second': 271.561,
 'eval_steps_per_second': 8.596,
 'epoch': 2.0}